In [ ]:
import bw2data as bd
import bw2io as bi
import bw2calc as bc
import bw2analyzer as bwa
import polars as pl 

In [ ]:
bd.projects.set_current("2026-01-13-ecoinvent-3.11-cutoff")

In [ ]:
bd.databases

In [ ]:
arch_mi_db = bd.Database("2026-2-12-10h51_archetypes_RE2020_MI")
arch_bb_db = bd.Database("2026-2-12-15h17_archetypes_RE2020_BB")


In [ ]:
archs = [a['name'] for a in arch_bb_db]
dbs = [arch_bb_db,arch_mi_db]
indicators = [m for m in bd.methods if "ef v3.1" in str(m).lower() and "no lt" not in str(m).lower()]
df_res = pl.DataFrame(schema={"usage" : pl.String,"type_archetype" : pl.String,"indicateur" : pl.List(pl.String),"value" : pl.Float64})

In [ ]:
from itertools import product

dbs = {
    "BB" : arch_bb_db,
    "MI" : arch_mi_db
}
demands = {
    f"{type_arch}@{usage}" : {dbs[type_arch].search(usage)[0].id : 1.0 }
    for type_arch, usage in  product(["BB","MI"], archs)
}
config ={"impact_categories" : indicators}

In [ ]:

data_objs = bd.get_multilca_data_objs(functional_units=demands,method_config=config)
lca=bc.MultiLCA(demands=demands, method_config=config, data_objs=data_objs)
lca.lci()
lca.lcia()
res = lca.scores

In [ ]:
for k,v in res.items():
    arch = k[1].split("@")

    new_row = pl.DataFrame(
        {
            "usage": [arch[1]],
            "type_archetype": [arch[0]],
            "indicateur": [k[0]],
            "value" : v
        }
        )
    df_res.extend(new_row)


In [ ]:
df_res = df_res.with_columns(
    pl.col("indicateur")
    .map_elements(
        lambda x : str(list(x)),
        return_dtype = pl.String
    ).alias("indicateur_str")
)

In [ ]:
with pl.Config(tbl_rows=-1):
    print(df_res.filter((pl.col("indicateur_str").str.contains("GWP")) & (pl.col("indicateur_str").str.contains("total"))))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
fig,ax = plt.subplots()
sns.scatterplot(data = df_res.filter(
    (pl.col("indicateur_str").str.contains("GWP")) 
    & (pl.col("indicateur_str").str.contains("total"))
    ).to_pandas(),
    x="type_archetype",
    y="value",
    hue="usage"
    )

# Comparaison archétypes buildings based vs scores bdd

In [1]:
# On récupère données
from archetypes_re2020.archetypes.buildings_based_archetypes_db import load_parquet_data, INPUT_PATH
from archetypes_re2020.data.source import INPUT_PATHS
df_re2020 = load_parquet_data(INPUT_PATH)


19:03:10+0100 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
df_re2020.columns

['projet_id',
 'batiment_index',
 'zone_index',
 'lot_ref',
 'sous_lot_ref',
 'type_donnees',
 'type_declaration',
 'unite_uf_txt',
 'nomenclature_inies_1',
 'nomenclature_inies_2',
 'nomenclature_inies_3',
 'nomenclature_inies_4',
 'nomenclature_inies_5',
 'dve',
 'quantite',
 'nomenclatures',
 'usage_principal_1_txt',
 'sref',
 'puissance_thermique_max_restituable_ch_1',
 'puissance_thermique_max_restituable_ch_2',
 'quant_surf',
 'nomenclatures_UF',
 'nomenclatures_UF_dve']

In [ ]:
from archetypes_re2020.archetypes.buildings_based_archetypes_db import load_parquet_data, INPUT_PATH
from archetypes_re2020.data.source import INPUT_PATHS

df_buildings = pl.read_csv(
    source=INPUT_PATHS["BUILDINGS"], ignore_errors=True
    )

In [ ]:
gwps = [m for m in bd.methods if "gwp" in str(m).lower() and "no LT" not in str(m) and "EF v3.1" in str(m) and "fossil" not in str(m) and "biogenic" not in str(m) and "land" not in str(m)]
gwps

In [ ]:
def ic_composant_per_use(df) :
    usages = df.select(
        pl.col("usage_principal_1_txt").unique()
        ).drop_nulls()["usage_principal_1_txt"]


    return {
        u : df.filter(
            pl.col("usage_principal_1_txt")==u
            ).select("projet_id","batiment_index","ic_composant").unique()["ic_composant"]
            for u in usages
            }

In [ ]:
dict_ic_composant = ic_composant_per_use(df_buildings)

In [ ]:
df_ic_composant = pl.DataFrame(schema={"usage" : pl.String,"value" : pl.Float64,"indicateur" : pl.String})
for u,v in dict_ic_composant.items() :
    for value in v :
        new_row = pl.DataFrame(
            {
                "usage": u,
                "value" : value,
                "indicateur" : "RE2020"
            }
            )
        df_ic_composant.extend(new_row)

In [ ]:
def chunk_list(lst, chunk_size):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

In [ ]:
#On ajoute les résultats de calcul
from tqdm import tqdm
buildings_db = bd.Database("2026-3-19-11h27_buildings_RE2020")
demands = {
    b["name"] : {b.id : 1.0 }
    for b in buildings_db
    }

config ={"impact_categories" : gwps}

In [ ]:
import gc
buildings_res = {}
config ={"impact_categories" : gwps}

for chunk in tqdm(chunk_list(list(buildings_db),2000)):
    demands = {
        b["name"] : {b.id : 1.0 }
        for b in chunk
        }
    data_objs = bd.get_multilca_data_objs(
        functional_units=demands,
        method_config=config
        )
    lca = bc.MultiLCA(demands=demands, method_config=config, data_objs=data_objs)
    lca.lci()
    lca.lcia()
    buildings_res |= lca.scores
    del lca
    del data_objs
    del demands
    gc.collect()

In [ ]:
from datetime import datetime
today = datetime.today()

In [ ]:
import pickle
with open(f"/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/processed/{today.year}-{today.month}-{today.day}_res_per_building.pkl", "wb") as f:
    pickle.dump(buildings_res, f)

In [ ]:
init = buildings_db.random()
lca = bc.LCA(demand={init : 1.0}, method=gwps[0])
lca.lci()
lca.lcia()
buildings_res = { k : {} for k in gwps}
for ind in gwps : 
    init = buildings_db.random()
    lca = bc.LCA(demand={init : 1.0}, method= ind)
    lca.lci()
    lca.lcia()
    for k, d in tqdm(demands.items(),desc = f"Calculs pour {ind}") :
        lca.redo_lcia(demand = d)
        buildings_res[ind][k] = lca.score



Import des résultats

In [ ]:
import pickle
with open("/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/processed/2026-3-19_res_per_building.pkl", "rb") as f:
    buildings_res = pickle.load(f)

In [ ]:
usage_dict = {a["name"] : a["usage"] for a in buildings_db}

In [ ]:
for k,v in tqdm(buildings_res.items()) :
    
    new_row = pl.DataFrame(
    {
        "usage": usage_dict[k[1]],
        "value" : v,
        "indicateur" : str(k[0])
    }
    )
    df_ic_composant.extend(new_row)

In [ ]:
df_ic_composant["usage"].unique()

# Détection des outliers à retirer

On utilise le critère de Chauvenet évoqué dans Heijungs 2024 7.8

In [ ]:
from scipy import stats
X = stats.Uniform(a=-0.5, b=0.5)
X.icdf(0.25)

In [ ]:
len(df_ic_composant.filter(pl.col("usage") == "Logement collectif", pl.col("indicateur") == "RE2020").drop_nulls())

In [ ]:
def chauvenet_dist(df,value_col) :
    mean = df[value_col].mean()
    std = df[value_col].std()
    n = len(df)
    X = stats.Normal(mu=mean,sigma=std)
    dmax = X.icdf(1/(4*n))
    print(mean,std,n,dmax)
    df = df.with_columns(
        chauvenet_dist = np.abs(pl.col(value_col) - mean)/std
    )
    df = df.with_columns(
        chauvenet_suspicious = pl.col('chauvenet_dist') > dmax
    )

    return df


In [ ]:
df_ic_composant_filtered = pl.DataFrame(schema={"usage" : pl.String,"value" : pl.Float64,"indicateur" : pl.String,"chauvenet_dist" : pl.Float64,"chauvenet_suspicious" : pl.Boolean})

for usage in df_ic_composant["usage"].drop_nulls().unique() :
    for ind in df_ic_composant["indicateur"].unique() :
        tmp = df_ic_composant.filter(
            pl.col("usage") == usage,
            pl.col("indicateur") == ind,
        )
        df_tmp = chauvenet_dist(tmp,"value")

        df_ic_composant_filtered = pl.concat([df_ic_composant_filtered,df_tmp])
        

In [ ]:
df_ic_composant_filtered.filter(pl.col("usage") == "Maison individuelle ou accolée",pl.col("indicateur") == "RE2020",pl.col("chauvenet_suspicious")==True)

Ca ne marche pas car les moyennes sont trop décalées par les valeurs aberrantes. On se rabat sur une méthode plus classique d'interquartile.

In [ ]:
df_ic_composant.filter(pl.col("usage") == "Maison individuelle ou accolée",pl.col("indicateur") == "RE2020").quantile(0.25,"nearest")['value'][0]

In [ ]:
def outliers(df,value_col) :
    mean = df[value_col].mean()
    std = df[value_col].std()
    q1 = df.quantile(0.25, "nearest")['value'][0]
    q3 = df.quantile(0.75, "nearest")['value'][0]
    iqr = q3 - q1
    low_mild = q1-1.5*iqr
    low_extreme = q1-3*iqr
    high_mild = q3+1.5*iqr
    high_extreme = q3+3*iqr

    df = df.with_columns(
        mild_outliers = (
            (
            (pl.col(value_col) < low_mild) 
            & (pl.col(value_col) >= low_extreme)
            ) | (
            (pl.col(value_col) > high_mild )
            & (pl.col(value_col) <= high_extreme)
            )
            )
    )
    df = df.with_columns(
        extreme_outliers = (
            (pl.col(value_col)<low_extreme)
            | (pl.col(value_col)>high_extreme)
        )
    )

    return df

In [ ]:
df_ic_composant_filtered = pl.DataFrame(schema={"usage" : pl.String,"value" : pl.Float64,"indicateur" : pl.String,"mild_outliers" : pl.Boolean,"extreme_outliers" : pl.Boolean})

for usage in df_ic_composant["usage"].drop_nulls().unique() :
    for ind in df_ic_composant["indicateur"].unique() :
        tmp = df_ic_composant.filter(
            pl.col("usage") == usage,
            pl.col("indicateur") == ind,
        ).drop_nulls()
        df_tmp = outliers(tmp,"value")

        df_ic_composant_filtered = pl.concat([df_ic_composant_filtered,df_tmp])
        

In [ ]:
df_ic_composant_filtered.filter(pl.col("usage") == "Maison individuelle ou accolée",pl.col("indicateur") == "RE2020",pl.col("extreme_outliers")== True)



In [ ]:
import numpy as np

In [ ]:
range_sizes= list(range(10**10,10**3,-10**4))


In [ ]:
base = (
    df_ic_composant
    .filter(
        (pl.col("indicateur") == "RE2020") & 
        (pl.col("usage") == "Maison individuelle ou accolée")
    )
    .select("value")
    .drop_nulls()
    .to_series()
)

med = base.median()
stds = {size : base
    .filter((base >= med - size/2) & (base <= med + size/2))
    .mean()
    for size in tqdm(range_sizes)
    }

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.lineplot(x= list(stds.keys()),y = list(stds.values()) )
# plt.xscale('log')
# plt.yscale('log')
# plt.xlim(0,100000)
# plt.ylim(0,200)

In [ ]:
range_sizes = [
    10**13,5*10**12,10**12,7*10**11,10**11,10**7,10**6,
    *[i*10**5 for i in range(1,9)],
    *[i*10**4 for i in range(1,9)],
    *[i*10**3 for i in range(1,9)],
    *[i*10**2 for i in range(1,9)],
    *[i*10**1 for i in range(1,9)],
    ]

In [ ]:
size =3*10**4

base.filter((base >= med - size/2) & (base <= med + size/2)).std()

In [ ]:
df_ic_composant.filter(pl.col("value")<0).select("value").describe().filter(pl.col("statistic") == "min")["value"][0]

In [ ]:
import great_tables as gt
import pandas as pd

In [ ]:
to_stats = df_ic_composant_filtered.filter(pl.col("extreme_outliers")== False,pl.col("mild_outliers")== False)

In [ ]:
moments = list(to_stats.describe()["statistic"])

df_ic_composant_stats = pl.DataFrame(schema={
    "usage" : pl.String,
    "indicateur" : pl.String,
    **{k: pl.Float64 for k in moments}
    }
    )

for usage in to_stats["usage"].drop_nulls().unique() :
    for ind in to_stats["indicateur"].unique() :
        tmp = to_stats.filter(
            pl.col("usage") == usage,
            pl.col("indicateur") == ind,
        ).select("value").describe()

        new_row = pl.DataFrame({
            "usage": usage,
            "indicateur" : ind,
            **{ k : tmp.filter(
                pl.col("statistic") == k
                )["value"][0] for k in moments}
            }
            )
        df_ic_composant_stats.extend(new_row)
        

In [ ]:
# import re
# def parse_col(col_name):
#     parts = re.findall(r'"((?:[^"\\]|\\.)*)"', col_name)
#     return tuple(parts)

to_table=df_ic_composant_stats.unpivot(on = moments,index =["usage","indicateur"],variable_name= "statistique")

to_table = to_table.pivot(on=["usage","indicateur"],index = ["statistique"])
to_table = to_table.to_pandas()
stat_series = to_table["statistique"]
cols = [c for c in to_table.columns if c !="statistique"]
tuples_cols = [parse_col(c) for c in cols]
multi_index = pd.MultiIndex.from_tuples(tuples_cols, names =["usage","indicateur"])
df_multi = to_table[cols].copy()
df_multi.columns = multi_index
df_multi.index = stat_series
df_multi.index.name = "statistique"
df_multi.to_excel("/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/reports/tables/2026-2-20_stats_wo_mild_and_extreme_outliers.xlsx")
# to_table

# Tracés

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import simplstyles
plt.style.use("nord-light")

In [ ]:
title = "comparaison_modele_vs_RE2020"

from datetime import datetime
today = datetime.today()




for usage in df_ic_composant_filtered['usage'].unique() :

    to_graph = df_ic_composant_filtered.filter(
        pl.col("extreme_outliers") == False,
        # pl.col("mild_outliers") == False,
        pl.col("usage") == usage
        ).to_pandas()

    usages = to_graph["usage"].unique()
    indicateurs = to_graph["indicateur"].unique()
    n_ind = len(indicateurs)

    hue_order = sorted(indicateurs)  # seaborn sorts hue by default
    x_positions = {usage: i for i, usage in enumerate(sorted(usages))}

    fig, ax = plt.subplots(figsize = (15,7))

    sns.boxenplot(data=to_graph, 
        x = "usage",
        y = "value",
        hue = "indicateur",
        gap=.1,
        hue_order = hue_order
        )

    # plt.xticks(rotation=45, ha='right');
    plt.style.use("nord-light")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5,1.1))


    width = 0.8  
    offsets = [(j - (n_ind - 1) / 2) * (width / n_ind) for j in range(n_ind)]

    for j, ind in enumerate(hue_order):
        for usage in sorted(usages):
            subset = to_graph[
                (to_graph["usage"] == usage) &
                (to_graph["indicateur"] == ind)
            ]["value"].dropna()

            if len(subset) == 0:
                continue

            mean_val = subset.mean()
            median_val = subset.median()
            std_val = subset.std()

            x = x_positions[usage] + offsets[j]
            y_max = 1.03*subset.max() # place text just above 99th percentile

            ax.text(
                x, y_max,
                f"N={len(subset)}\nμ={mean_val:.0f}\nmed={median_val:.0f}\nσ={std_val:.0f}",
                ha='center', va='bottom',
                fontsize=9,
                color="#4c566a",
                linespacing=1.4
            )


    plt.savefig(
        f"/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/reports/figures/{today.year}-{today.month}-{today.day}_{title}_{usage}.pdf",
        dpi=300,
        bbox_inches = "tight"
        )

# Tracé des différences

In [ ]:
to_compare = df_buildings.select("projet_id","batiment_index","ic_composant").unique()
df_ic_composant_diff = pl.DataFrame(schema={"usage" : pl.String,"value" : pl.Float64,"indicateur" : pl.String})

for k,v in tqdm(buildings_res.items()) :
    b_id = float(k[1].split("_")[1])
    p_id = float(k[1].split("_")[0])
    try :
        value_re2020 = to_compare.filter(pl.col("projet_id") == p_id, pl.col("batiment_index")==b_id)["ic_composant"][0]
        new_row = pl.DataFrame(
        {
            "usage": usage_dict[k[1]],
            "value" : v-value_re2020,
            "indicateur" : f"{str(k[0])} - RE2020"
        }
        )
        df_ic_composant_diff.extend(new_row)
    except IndexError :
        # print(f"Problème avec {k}")
        continue

In [ ]:

df_ic_composant_diff_filtered = pl.DataFrame(schema={"usage" : pl.String,"value" : pl.Float64,"indicateur" : pl.String,"mild_outliers" : pl.Boolean,"extreme_outliers" : pl.Boolean})

for usage in df_ic_composant_diff["usage"].drop_nulls().unique() :
    for ind in df_ic_composant_diff["indicateur"].unique() :
        tmp = df_ic_composant_diff.filter(
            pl.col("usage") == usage,
            pl.col("indicateur") == ind,
        ).drop_nulls()
        df_tmp = outliers(tmp,"value")

        df_ic_composant_diff_filtered = pl.concat([df_ic_composant_diff_filtered,df_tmp])
        

In [ ]:
title = "comparaison_modele_vs_RE2020_difference"


from datetime import datetime
today = datetime.today()

for usage in df_ic_composant_diff_filtered["usage"].unique() :

    to_graph = df_ic_composant_diff_filtered.filter(
        pl.col("extreme_outliers") ==False,
        pl.col("usage") == usage
        ).to_pandas()


    usages = to_graph["usage"].unique()
    indicateurs = to_graph["indicateur"].unique()
    n_ind = len(indicateurs)

    # Map hue order to x-offset (mimics seaborn's internal dodge)
    hue_order = sorted(indicateurs)  # seaborn sorts hue by default
    x_positions = {usage: i for i, usage in enumerate(sorted(usages))}

    fig, ax = plt.subplots(figsize = (15,7))
    ax.axhline(0,color="#d08770")

    sns.boxenplot(data=to_graph, 
        x = "usage",
        y = "value",
        hue = "indicateur",
        hue_order = hue_order,
        order = list(sorted(usages)),
        gap=.1
        )

    plt.xticks(rotation=45, ha='right');
    plt.style.use("nord-light")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5,1.1))

    width = 0.8  
    offsets = [(j - (n_ind - 1) / 2) * (width / n_ind) for j in range(n_ind)]

    for j, ind in enumerate(hue_order):
        for usage in sorted(usages):
            subset = to_graph[
                (to_graph["usage"] == usage) &
                (to_graph["indicateur"] == ind)
            ]["value"].dropna()

            if len(subset) == 0:
                continue

            mean_val = subset.mean()
            median_val = subset.median()
            std_val = subset.std()

            x = x_positions[usage] + offsets[j]
            y_max = 1.05*subset.max()  # place text just above 99th percentile

            ax.text(
                x, y_max,
                f"N={len(subset)}\nμ={mean_val:.0f}\nmed={median_val:.0f}\nσ={std_val:.0f}",
                ha='center', va='bottom',
                fontsize=9,
                color="#4c566a",
                linespacing=1.4
            )

    plt.savefig(
        f"/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/reports/figures/{today.year}-{today.month}-{today.day}_{title}_{usage}.pdf",
        dpi=300,
        bbox_inches = "tight"
        )

In [ ]:
list(sorted(usages))

In [ ]:
del bd.databases["2026-2-12-10h23_batiments_RE2020_wo_pump"]

In [ ]:
bd.databases

In [ ]:
# On veut faire le même graphique, sans les pompes à chaleur

buildings_db_wo_pump = bd.Database("2026-2-12-10h23_batiments_RE2020").copy("2026-2-12-10h23_batiments_RE2020_wo_pump")

In [ ]:
buildings_db_wo_pump = bd.Database("2026-2-12-10h23_batiments_RE2020_wo_pump")

In [ ]:
a = buildings_db_wo_pump.random()

In [ ]:
from tqdm import tqdm

for b in tqdm(buildings_db_wo_pump) :
    exch = [e for e in b.edges() if "thermodynamique" in e["input"][1].lower()]
    if exch :
        for e in exch : 
            e.delete()

In [ ]:
import gc
buildings_res_wo_pump = {}
config ={"impact_categories" : gwps}

for chunk in tqdm(chunk_list(list(buildings_db_wo_pump),2000)):
    demands = {
        b["name"] : {b.id : 1.0 }
        for b in chunk
        }
    data_objs = bd.get_multilca_data_objs(
        functional_units=demands,
        method_config=config
        )
    lca = bc.MultiLCA(demands=demands, method_config=config, data_objs=data_objs)
    lca.lci()
    lca.lcia()
    buildings_res_wo_pump |= lca.scores
    del lca
    del data_objs
    del demands
    gc.collect()

import pickle
today = datetime.today()
with open(f"/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/processed/{today.year}-{today.month}-{today.day}_res_per_building_wo_pump.pkl", "wb") as f:
    pickle.dump(buildings_res, f)

In [ ]:
with open("/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/processed/2026-2-18_res_per_building_wo_pump.pkl", "rb") as f:
    buildings_res_wo_pump = pickle.load(f)
    

In [ ]:
to_compare = df_buildings.select("projet_id","batiment_index","ic_composant").unique()
df_ic_composant_diff_wo_pump = pl.DataFrame(schema={"usage" : pl.String,"value" : pl.Float64,"indicateur" : pl.String})

for k,v in tqdm(buildings_res_wo_pump.items()) :
    b_id = float(k[1].split("_")[1])
    p_id = float(k[1].split("_")[0])
    try :
        value_re2020 = to_compare.filter(pl.col("projet_id") == p_id, pl.col("batiment_index")==b_id)["ic_composant"][0]
        new_row = pl.DataFrame(
        {
            "usage": usage_dict[k[1]],
            "value" : v-value_re2020,
            "indicateur" : f"{str(k[0])} - RE2020"
        }
        )
        df_ic_composant_diff_wo_pump.extend(new_row)
    except IndexError :
        # print(f"Problème avec {k}")
        continue

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
title = "comparaison_modele_vs_RE2020_difference_wo_pump"
cap_up = 2000

cap_down = -2000


from datetime import datetime
today = datetime.today()

to_graph = df_ic_composant_diff_wo_pump.filter(
    (pl.col("value")>-2000)
    & (pl.col("value")<2000)
    ).to_pandas()

usages = to_graph["usage"].unique()
indicateurs = to_graph["indicateur"].unique()
n_ind = len(indicateurs)

# Map hue order to x-offset (mimics seaborn's internal dodge)
hue_order = sorted(indicateurs)  # seaborn sorts hue by default
x_positions = {usage: i for i, usage in enumerate(sorted(usages))}

fig, ax = plt.subplots(figsize = (15,7))
ax.axhline(0,color="#d08770")

sns.boxenplot(data=to_graph, 
    x = "usage",
    y = "value",
    hue = "indicateur",
    hue_order = hue_order,
    order = list(sorted(usages)),
    gap=.1
    )

plt.xticks(rotation=45, ha='right');
plt.style.use("nord-light")
ax.legend(loc="lower center", bbox_to_anchor=(0.5,1.1))

width = 0.8  
offsets = [(j - (n_ind - 1) / 2) * (width / n_ind) for j in range(n_ind)]

for j, ind in enumerate(hue_order):
    for usage in sorted(usages):
        subset = to_graph[
            (to_graph["usage"] == usage) &
            (to_graph["indicateur"] == ind)
        ]["value"].dropna()

        if len(subset) == 0:
            continue

        mean_val = subset.mean()
        median_val = subset.median()
        std_val = subset.std()

        x = x_positions[usage] + offsets[j]
        y_max = 1.1*cap_up  # place text just above 99th percentile

        ax.text(
            x, y_max,
            f"μ={mean_val:.0f}\nmed={median_val:.0f}\nσ={std_val:.0f}",
            ha='center', va='bottom',
            fontsize=9,
            color="#4c566a",
            linespacing=1.4
        )
plt.savefig(
    f"/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/reports/figures/{today.year}-{today.month}-{today.day}_{title}.pdf",
    dpi=300,
    bbox_inches = "tight"
    )

In [ ]:
a = buildings_db.random()
lca = bc.LCA(demand={a:1.0},method=gwps[1])
lca.lci()
lca.lcia()

In [ ]:
mat_0 = lca.characterization_matrix

In [ ]:
mat_1 = lca.characterization_matrix

In [ ]:
mat_EF31 = mat_0
mat_15804 = mat_1

In [ ]:
# EN15804 + A2 - EF3.1
diff = mat_1-mat_0

In [ ]:
print(diff)

In [ ]:
bd.get_node(id = lca.dicts.biosphere.reversed[2526])

In [ ]:
gwps[0]

In [ ]:
mat_1

In [ ]:
mat_15804[1424,1424]

In [ ]:
title = "comparaison_modele_vs_RE2020_difference"

from datetime import datetime
today = datetime.today()

to_graph = df_ic_composant.filter(
    (pl.col("value")>-2000)
    & (pl.col("value")<2000)
    & (pl.col("usage")=="Maison individuelle ou accolée")
    ).to_pandas()

fig, ax = plt.subplots(figsize = (15,7))
# ax.axhline(0,color="#d08770")
sns.histplot(data=to_graph, 
    x = "value",
    # y = "value",
    hue = "indicateur",
    # inner = None
    # inner_kws = dict(box_width=12, whis_width=2, color=(0,0,0))
    legend = True
    )
# sns.boxplot(data=to_graph, 
#     x = "usage",
#     y = "value",
#     hue = "indicateur",
#     gap=.1
#     )
plt.xticks(rotation=45, ha='right');
# plt.style.use("nord-light")
# ax.legend(loc="upper center", bbox_to_anchor=(0.5,1.1))

# plt.savefig(
#     f"/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/reports/figures/{today.year}-{today.month}-{today.day}_{title}.pdf",
#     dpi=300,
#     bbox_inches = "tight"
#     )

In [ ]:
id_ex = [k[1] for k,v in buildings_res.items() if v>0][0]

In [ ]:
b = buildings_db.search(id_ex)[0]

In [ ]:
list(b.edges())

In [ ]:
to_compare = df_buildings.select("projet_id","batiment_index","ic_composant").unique()
b_id = float(id_ex.split("_")[1])
p_id = float(id_ex.split("_")[0])

value_re2020 = to_compare.filter(pl.col("projet_id") == p_id, pl.col("batiment_index")==b_id)["ic_composant"][0]

In [ ]:
value_re2020

In [ ]:
df_buildings.filter(pl.col("projet_id")==p_id).columns

In [ ]:
bwa.print_recursive_calculation(activity=b,lcia_method=gwps[1]) 

In [ ]:
list(b.edges())

In [ ]:
df_re2020